In [ ]:
# Cell 1: 终极环境配置与 SparkSession 初始化
import os
import sys

# 1. Windows 平台特异性兼容配置 (Anti-Gravity 底层加固)
os.environ['_JAVA_OPTIONS'] = '-Djava.net.preferIPv4Stack=true'

hadoop_home = os.environ.get('HADOOP_HOME')
if not hadoop_home or not os.path.exists(hadoop_home):
    hadoop_home = r'C:\hadoop'
os.environ['HADOOP_HOME'] = hadoop_home

hadoop_bin = os.path.join(hadoop_home, 'bin')
if hadoop_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = hadoop_bin + ';' + os.environ.get('PATH', '')

# 锁死 Python 解释器路径
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession

# 2. 初始化具备强隔离与低开销的 SparkSession
spark = SparkSession.builder \
    .appName("UnsupervisedRiskAnomalyDetection") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true") \
    .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f"✅ SparkSession 启动成功! Spark Version: {spark.version}")

In [ ]:
# Cell 2: 加载特征库并构造交易数值特征宽表，使用 VectorAssembler 与 StandardScaler 预处理
from pyspark.sql.functions import col, when, rand, round
from pyspark.ml.feature import VectorAssembler, StandardScaler

# 1. 加载离线历史特征库
features_parquet_path = "../data/features/offline_risk_features.parquet"
offline_features = spark.read.parquet(features_parquet_path)
print(f"📋 离线特征库已加载，样本数: {offline_features.count()}")

# 2. 纯 JVM 声明式生成模拟多维交易行为宽表 (100% 避开 Windows Python Worker 通信缺陷)
# 模拟包含 12% 黑产团伙异常模式与 88% 正常基线
df_raw = spark.range(0, 300).select(
    col("id").cast("string").alias("device_id"),
    when(rand(42) > 0.88, 1).otherwise(0).alias("is_syndicate")
).withColumn(
    "order_freq",
    when(col("is_syndicate") == 1, round(rand(1) * 80 + 20, 1))
    .otherwise(round(rand(2) * 5 + 1, 1))
).withColumn(
    "total_amount",
    when(col("is_syndicate") == 1, round(rand(3) * 8000 + 2000, 2))
    .otherwise(round(rand(4) * 400 + 50, 2))
).withColumn(
    "coupon_count",
    when(col("is_syndicate") == 1, round(rand(5) * 30 + 10, 1))
    .otherwise(round(rand(6) * 3 + 1, 1))
).withColumn(
    "refund_count",
    when(col("is_syndicate") == 1, round(rand(7) * 15 + 5, 1))
    .otherwise(round(rand(8) * 1, 1))
).withColumn(
    "history_risk_score",
    when(col("is_syndicate") == 1, round(rand(9) * 50 + 80, 2))
    .otherwise(round(rand(10) * 30 + 10, 2))
)

feature_cols = ["order_freq", "total_amount", "coupon_count", "refund_count", "history_risk_score"]

# 3. 特征组装 (VectorAssembler)
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
df_assembled = assembler.transform(df_raw)

# 4. 标准化消除量纲影响 (StandardScaler: 均值归零，方差归一)
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print("📊 特征预处理宽表预览:")
df_scaled.select("device_id", "order_freq", "total_amount", "scaled_features").show(5, truncate=False)

In [ ]:
# Cell 3: KMeans 聚类与样本到簇中心欧氏距离 (KMeans Outlier Anomaly Score)
from pyspark.ml.clustering import KMeans
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, lit, sqrt, when

# 1. 训练 KMeans 聚类模型 (设定 k=3)
kmeans = KMeans(featuresCol="scaled_features", predictionCol="cluster", k=3, seed=42)
kmeans_model = kmeans.fit(df_scaled)
df_clustered = kmeans_model.transform(df_scaled)

# 2. 获取聚类中心坐标并转换为纯 Catalyst 算子计算距离 (避开 Python UDF 崩溃)
centers = kmeans_model.clusterCenters()
df_with_arr = df_clustered.withColumn("features_arr", vector_to_array(col("scaled_features")))

dist_expr = lit(0.0)
for k_idx, center in enumerate(centers):
    sq_dist_k = sum((col("features_arr")[dim_idx] - float(val)) ** 2 for dim_idx, val in enumerate(center))
    dist_expr = when(col("cluster") == k_idx, sqrt(sq_dist_k + 1e-12)).otherwise(dist_expr)

df_kmeans_scored = df_with_arr.withColumn("kmeans_anomaly_score", dist_expr)

print("🎯 KMeans 异常分打分结果:")
df_kmeans_scored.select("device_id", "cluster", "kmeans_anomaly_score").show(5, truncate=False)

In [ ]:
# Cell 4: PCA 主成分降维与重构误差计算 (Reconstruction Error Anomaly Score)
import numpy as np
from pyspark.ml.feature import PCA
from pyspark.sql.functions import col, lit, sqrt

# 1. 训练 PCA 降维模型 (保留前 2 个主成分)
pca = PCA(k=2, inputCol="scaled_features", outputCol="pca_features")
pca_model = pca.fit(df_kmeans_scored)
df_pca = pca_model.transform(df_kmeans_scored)

# 2. 提取投影矩阵并构建重构投影算子: P = V @ V.T (维度: num_features x num_features)
V = pca_model.pc.toArray()
P = np.dot(V, V.T)
num_feats = len(P)

# 3. 纯 Catalyst 表达式计算重构误差: ||x - x @ P||_2
recon_sq_error = lit(0.0)
for i in range(num_feats):
    x_hat_i = sum(col("features_arr")[j] * float(P[j, i]) for j in range(num_feats))
    recon_sq_error = recon_sq_error + (col("features_arr")[i] - x_hat_i) ** 2

df_pca_scored = df_pca.withColumn("pca_anomaly_score", sqrt(recon_sq_error + 1e-12))

print("🔍 PCA 重构误差打分结果:")
df_pca_scored.select("device_id", "kmeans_anomaly_score", "pca_anomaly_score").show(5, truncate=False)

In [ ]:
# Cell 5: 综合双异常分，生成半监督伪标签 (Top 5% 为 1, Bottom 50% 为 0) 并训练 RandomForest
from pyspark.sql.functions import col, when, min as spark_min, max as spark_max
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. 归一化双异常分并融合成综合异常评分 (0 ~ 1 范围)
stats = df_pca_scored.agg(
    spark_min("kmeans_anomaly_score").alias("k_min"),
    spark_max("kmeans_anomaly_score").alias("k_max"),
    spark_min("pca_anomaly_score").alias("p_min"),
    spark_max("pca_anomaly_score").alias("p_max")
).collect()[0]

k_min, k_max = stats["k_min"], stats["k_max"]
p_min, p_max = stats["p_min"], stats["p_max"]

norm_kmeans = (col("kmeans_anomaly_score") - k_min) / (k_max - k_min + 1e-6)
norm_pca = (col("pca_anomaly_score") - p_min) / (p_max - p_min + 1e-6)
df_combined = df_pca_scored.withColumn("combined_anomaly_score", (norm_kmeans + norm_pca) / 2.0)

# 2. 计算分位数切点: 50% 安全基线与 95% 高危切点
q50, q95 = df_combined.approxQuantile("combined_anomaly_score", [0.50, 0.95], 0.01)
print(f"📊 分位数切点: Bottom 50% 安全线 = {q50:.4f}, Top 5% 高危切线 = {q95:.4f}")

# 3. 打伪标签: Top 5% -> 1.0 (高危欺诈), Bottom 50% -> 0.0 (安全正常), 其余置空
df_labeled = df_combined.withColumn(
    "label",
    when(col("combined_anomaly_score") >= q95, 1.0)
    .when(col("combined_anomaly_score") <= q50, 0.0)
    .otherwise(None)
)

# 4. 提取伪标签训练集 (过滤掉置信度不足的中间 45% 模糊地带)
training_df = df_labeled.filter(col("label").isNotNull())
print(f"📈 伪标签有效训练样本数: {training_df.count()} (正样本: {training_df.filter(col('label') == 1.0).count()})")

# 5. 训练 RandomForestClassifier 分类器
rf = RandomForestClassifier(featuresCol="scaled_features", labelCol="label", numTrees=20, maxDepth=5, seed=42)
rf_model = rf.fit(training_df)
predictions = rf_model.transform(training_df)

# 6. 评估训练集分类表现
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"🎯 RandomForest 伪标签模型训练完成! ROC-AUC: {auc:.4f}")

# 7. 输出各维度特征对黑产判定的相对贡献权重
print("\n🔥 特征重要性贡献度:")
importances = rf_model.featureImportances
for feat_name, imp in zip(feature_cols, importances):
    print(f"  - {feat_name:20s}: {imp * 100:.2f}%")